# 02. Retention, re-access, and witnessed difference

**Level:** Intermediate

**Prerequisites:** Lessons 000-010; familiarity with hashes as content fingerprints.

**Canonical claim anchors:** `WDP-CH02-P01`, `WDP-CH02-N01`, `WDP-CH03-O02`

## Learning objectives

- construct a retained registration and content-bound witness
- distinguish a registration from later successful re-access
- interpret digest mismatch as a refusal rather than a false success

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope
boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: a mark is not yet a witness

A `Registration` records a mark, a carrier identifier, and an occurrence. A
`WitnessReceipt` additionally records a later evaluator re-accessing the same
canonical content. The two stages are deliberately separate: retained data
can fail to be re-accessed, and later content can differ from the retained
mark.

The digest supports exact replay of this finite software object. It does not
certify the truth of an external claim encoded in the mark.


## Visual checkpoint

![Witness Ladder](../images/generated/witness_ladder.svg)

These files are deterministic generated aids. Their captions and the executed data, not visual resemblance alone, carry the intended lesson.


## Derivation

The receipt digest is computed from the carrier, occurrence, retained flag,
evaluator, expected mark digest, and observed mark digest. Verification first
checks the supplied artifact against the observed digest and then recomputes
the receipt body. Thus changing either the artifact or receipt fields has a
named failure path.


In [ ]:
from witnessed_difference.notebook_support import display_json
from witnessed_difference.witness import Registration, make_witness, verify_witness

registration = Registration('lab-log', {'bit': 1, 'setting': 'A'}, 7)
made = make_witness(registration, {'setting': 'A', 'bit': 1}, 'evaluator-E')
witness = made.value
print(display_json({'registration': registration, 'witness': witness}))
print(display_json({
    'registration_type': type(registration).__name__,
    'witness_type': type(witness).__name__,
    'roles_are_not_coerced': type(registration) is not type(witness),
    'authority_effect': witness.authority_effect,
}))
print(display_json(verify_witness(witness, {'bit': 1, 'setting': 'A'})))


## Read the result

Key order does not alter the canonical mark digest, so the equivalent mapping
is accepted. The receipt still names one evaluator and one retained
occurrence. Successful replay is therefore narrower than observer-independent
truth or institutional validation.


In [ ]:
tampered = verify_witness(witness, {'bit': 0, 'setting': 'A'})
unretained = make_witness(Registration('scratch', 'mark', 0, retained=False), 'mark', 'E')
wrong_reaccess = make_witness(registration, {'bit': 1, 'setting': 'B'}, 'evaluator-E')
print(display_json({'tampered': tampered, 'unretained': unretained, 'wrong_reaccess': wrong_reaccess}))


## Exercise

Create a second valid witness for the same registration but with evaluator
`evaluator-F`. Compare the two receipt digests. Explain why equal observed
content does not make the two complete receipts identical.


In [ ]:
second = make_witness(registration, {'bit': 1, 'setting': 'A'}, 'evaluator-F').value
print(display_json({
    'same_mark_digest': second.registration.mark_digest == witness.registration.mark_digest,
    'same_receipt_digest': second.receipt_digest == witness.receipt_digest,
    'reason': 'the evaluator identity is part of the receipt body',
}))


## Scope check

A replayable hash binds declared bytes and fields; it does not establish
semantic correctness, phenomenology, or authorization. **AuthorityEffect:
None.**


## Continue the learning path

- Previous: [Cochains, exactness, and the vector obstruction](010_cochains_and_vector_obstruction.ipynb)
- Index: [Notebook curriculum](README.md)
- Next: [Three observer representations and explicit bridges](030_observer_representations.ipynb)
